# Unidade 4 - Bloco prático da Aula 01: CASH artesanal com Optuna

Monta à mão um espaço CASH: suggest_categorical escolhe o algoritmo e cada ramo abre os hiperparâmetros daquele algoritmo, buscado com TPE. Acompanhe como o orçamento de tentativas se concentra no ramo mais promissor ao longo da busca.

In [ ]:
import numpy as np
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              GradientBoostingClassifier)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)
X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objetivo(trial):
    algo = trial.suggest_categorical(
        "algoritmo", ["logistica", "floresta", "boosting"])
    if algo == "logistica":
        modelo = make_pipeline(StandardScaler(), LogisticRegression(
            C=trial.suggest_float("logreg_C", 1e-3, 100, log=True),
            max_iter=2000))
    elif algo == "floresta":
        modelo = RandomForestClassifier(
            n_estimators=trial.suggest_int("rf_n_estimators", 50, 300),
            max_depth=trial.suggest_int("rf_max_depth", 3, 15),
            min_samples_leaf=trial.suggest_int("rf_min_leaf", 1, 10),
            random_state=42, n_jobs=-1)
    else:
        modelo = GradientBoostingClassifier(
            n_estimators=trial.suggest_int("gb_n_estimators", 50, 300),
            learning_rate=trial.suggest_float("gb_lr", 1e-3, 0.3, log=True),
            max_depth=trial.suggest_int("gb_max_depth", 2, 5),
            random_state=42)
    return cross_val_score(modelo, X, y, cv=cv, scoring="f1",
                           n_jobs=-1).mean()

estudo = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=42))
estudo.optimize(objetivo, n_trials=60)

print(f"Melhor F1: {estudo.best_value:.4f}")
print(f"Melhor configuracao: {estudo.best_params}")
from collections import Counter
contagem = Counter(t.params["algoritmo"] for t in estudo.trials)
print("Tentativas por algoritmo:", dict(contagem))